# 03 — Amostra 10% estratificada por década (pilot)

Gera `data/interim/metadados_amostra10.parquet` com 10% do corpus IPEA,
estratificado por década de publicação. Objetivo: rodar o pipeline
end-to-end (download + Docling + classificador LLM) num subset barato e
representativo antes de escalar para os 17.943 documentos.

- `random_state=42` fixo para reprodutibilidade.
- Descarta documentos sem ano ou com ano fora de `[1950, 2030]`
  (um outlier literal `ano=88` existe no corpus).

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))
from src.scraping import stratified_sample_by_decade

SRC = Path('../data/interim/metadados.parquet')
DEST = Path('../data/interim/metadados_amostra10.parquet')
df = pd.read_parquet(SRC)
print(f'Corpus completo: {len(df):,}')

Corpus completo: 17,943


In [2]:
amostra = stratified_sample_by_decade(df, frac=0.10, random_state=42)
amostra.drop(columns=['decada']).to_parquet(DEST, index=False)
print(f'Amostra 10%: {len(amostra):,} ({len(amostra)/len(df):.1%} do corpus)')
print(f'Escrita em {DEST}')

Amostra 10%: 1,758 (9.8% do corpus)
Escrita em ../data/interim/metadados_amostra10.parquet


## Integridade da estratificação

Compara a distribuição por década antes e depois do sample — ideal é manter
as proporções.

In [3]:
pop = df.dropna(subset=['ano']).copy()
pop = pop[(pop['ano'] >= 1950) & (pop['ano'] <= 2030)]
pop['decada'] = (pop['ano'].astype(int) // 10) * 10

comp = pd.DataFrame({
    'N_pop': pop.groupby('decada').size(),
    'N_amostra': amostra.groupby('decada').size(),
})
comp['frac_real'] = comp['N_amostra'] / comp['N_pop']
comp

,N_pop,N_amostra,frac_real
decada,,,
1960,250,25,0.100000
1970,836,84,0.100478
1980,1268,127,0.100158
1990,1855,186,0.100270
2000,2771,277,0.099964
2010,6086,609,0.100066
2020,4500,450,0.100000


## Cobertura da amostra

In [4]:
print(f'Com handle:  {amostra["handle"].notna().sum():>5} ({amostra["handle"].notna().mean():.1%})')
print(f'Com resumo:  {(amostra["resumo"] != "").sum():>5} ({(amostra["resumo"] != "").mean():.1%})')
print()
print('=== Top 10 tipos na amostra ===')
print(amostra['tipo'].fillna('(sem tipo)').value_counts().head(10).to_string())

Com handle:   1755 (99.8%)
Com resumo:   1757 (99.9%)

=== Top 10 tipos na amostra ===
tipo
Journal article            535
Working paper              485
Book                       262
Book part                  204
Journal                    101
Report                      89
Journal Article             57
                            18
Eventos                      5
Preliminary Publication      1


In [5]:
pivot = amostra.groupby(['decada', 'tipo']).size().unstack(fill_value=0)
pivot = pivot.loc[:, pivot.sum().sort_values(ascending=False).head(8).index]
print('=== Tipo x década (amostra 10%) ===')
pivot

=== Tipo x década (amostra 10%) ===


tipo,Journal article,Working paper,Book,Book part,Journal,Report,Journal Article,
decada,,,,,,,,
1960,0,0,23,0,0,2,0,0
1970,25,3,37,5,4,10,0,0
1980,39,22,57,2,0,7,0,0
1990,24,49,70,13,9,6,7,6
2000,97,89,23,32,21,9,6,0
2010,196,196,43,76,54,30,6,5
2020,154,126,9,76,13,25,38,7


## Próximo passo

Partir para a Fase 2 usando `data/interim/metadados_amostra10.parquet` como
input: download de PDF via `baixar_pdf_real` (port do IpeaPub), extração
com Docling (`do_ocr=True`, `do_table_structure=True`), saída em
`data/interim/textos_amostra10.parquet`.